# Имагология · блокнот 3. Параллельное чтение

| | |
|---|---|
| **Вопрос** | Одинаков ли Восток у русского и западного автора: какие мотивы и признаки выходят на первый план? |
| **Что вы получите** | Профили мотивов и признаков двух корпусов на 10 000 слов, «сцену» в обоих текстах, ключевые слова для текстов одного языка. |
| **Стенд в браузере** | [Параллельный читатель](https://pavelaleks.github.io/samarkand_lectures/imagology/lab/parallel.html) |
| **Практические** | 1, 3, 6, 7 · [задания](https://pavelaleks.github.io/samarkand_lectures/imagology/lab/practicum.html) |

**Правило курса:** описываем образ в тексте, а не «правду» о народе. Число без контекста в работу не идёт: каждую цифру проверьте предложением.

### Как работать
1. **Файл → Сохранить копию на Диске**, иначе правки пропадут.
2. Запускайте ячейки **сверху вниз** кнопкой ▶. Код спрятан, меняются только **поля формы** справа.
3. Сначала запустите всё как есть: демо на готовой паре текстов из Викитеки. Потом впишите свои тексты.

**Как задать тексты** (в полях через `;`):
- ключ готового текста: `pushkin-arzrum`, `custine`, `vambery` — список выводит `list_texts()`; русский перевод западного текста — `hugo-orientales@ru`;
- страница Викитеки: `ws:ru:Мцыри (Лермонтов)`, `ws:en:The Giaour`, `ws:fr:Les Orientales`;
- ссылка на .txt: `https://www.gutenberg.org/cache/epub/41751/pg41751.txt`;
- `МОИ ФАЙЛЫ` — появится кнопка загрузки .txt или .docx.

> Ошибка? **Среда выполнения → Перезапустить сеанс**, затем ячейки сверху вниз. Викитека иногда отвечает медленно: подождите и запустите ячейку ещё раз.

In [ ]:
#@title 🔧 Подготовка — просто запустите ▶ { display-mode: "form" }
import subprocess, sys, importlib, requests
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "python-docx", "folium"], check=False)
_u = "https://pavelaleks.github.io/samarkand_lectures/imagology/lab/colab/imago_utils.py"
open("imago_utils.py", "wb").write(requests.get(_u, timeout=60).content)
import imago_utils; importlib.reload(imago_utils)
from imago_utils import *
setup()

In [ ]:
#@title 📂 Тексты { display-mode: "form" }
#@markdown Готовая пара заполнит оба корпуса (поля ниже тогда не нужны). Список пар — `list_pairs()`.
PAIR = "orientales"  #@param ["—", "persidskie", "custine", "orientales", "divan", "itineraire", "arzrum", "plennik", "bela", "peri", "prorok", "spor", "ammalat", "burden", "samarkand", "sarty", "nikitin"]
#@markdown Корпус A (обычно Россия) и корпус B (обычно Запад): ключи, `ws:язык:Название`, ссылки, `МОИ ФАЙЛЫ` — через «;».
CORPUS_A = "pushkin-fontan"  #@param {type:"string"}
CORPUS_B = "hugo-orientales"  #@param {type:"string"}

if PAIR != "—":
    _p = next(p for p in imago_utils._data("presets")["pairs"] if p["id"] == PAIR)
    CORPUS_A, CORPUS_B = _p["left"], _p["right"]
    print("Пара:", _p["title"]); print("Вопрос:", _p["question"])
A = analyze_all(load_spec(CORPUS_A, "россия")) if CORPUS_A.strip() else []
B = analyze_all(load_spec(CORPUS_B, "запад")) if CORPUS_B.strip() else []
print("\nКорпус A:", "; ".join(a["label"] for a in A)); print("Корпус B:", "; ".join(b["label"] for b in B))

In [ ]:
#@title 💡 Гипотеза — до подсчёта! { display-mode: "form" }
HYPOTHESIS = "У Гюго мотив гарема и оружия заметнее, у Пушкина — сад и фонтан"  #@param {type:"string"}
print("Гипотеза зафиксирована:", HYPOTHESIS)

In [ ]:
#@title 📊 Профили { display-mode: "form" }
KIND = "motifs"  #@param ["motifs", "traits", "groups"]
PR = compare_profiles(A, B, KIND, names=("A", "B"))
plot_profiles(PR, {"motifs": "Мотивы", "traits": "Признаки", "groups": "Народы"}[KIND] + ", на 10 000 слов")
import matplotlib.pyplot as plt; plt.savefig("profili.png", dpi=200, bbox_inches="tight")
PR

In [ ]:
#@title 🎭 Сцена в обоих текстах { display-mode: "form" }
#@markdown Часть названия мотива: `гарем`, `кинжал`, `сад`, `пустын`, `мечет`.
MOTIF = "гарем"  #@param {type:"string"}
_m = find_id("motifs", MOTIF)
SC = contexts(A + B, motif=_m)
for side, g in SC.groupby("сторона"):
    print(f"\n— {side}: {len(g)} предложений")
    for s in g["предложение"].head(8):
        print("  •", s[:300])

In [ ]:
#@title 🔑 Ключевые слова (если оба текста на одном языке) { display-mode: "form" }
KA, KB = keyness(A, B)
print("Чаще в A:"); display(KA.head(20)) if not KA.empty else print("—")
print("Чаще в B:"); display(KB.head(20)) if not KB.empty else print("—")

In [ ]:
#@title 💾 Сохранить { display-mode: "form" }
save_excel({"Профили": PR, "Сцена": SC}, "parallel.xlsx")
download("parallel.xlsx"); download("profili.png")